In [ ]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.append("../src")
from busqueda import compuerta
from datos import cargar, gps_evento
from filtro_adaptado import chi2, psd_en, serie_snr, snr_optimo, snr_red, snr_reponderado, transformada
from plantillas import generador
from senal import irfft

In [ ]:
EVENTOS = ["GW150914", "GW151012", "GW151226", "GW170104", "GW170608", "GW170814", "GW170823"]
DATOS = Path("../datos")
DURACION = 128
DURACION_FFT = 4
DURACION_SEGMENTO = 32
F_MIN = 20
F_MAX = 1024
FS_SALIDA = 2048
INICIO_EVENTO = -24
BORDE_INICIAL = 13
BORDE_FINAL = 4
BANDAS_CHI2 = 16
TANDA = 50
REALIZACIONES = 20
SEPARACION = 41
MASAS_RUIDO = [30, 6]
INICIO_INYECCIONES = -64
MARGEN_INYECCION = 2
F_MIN_INYECCION = 15
SNR_INYECTADOS = [8, 16, 32]
INYECCIONES_POR_SNR = 10
MASA_MINIMA = 6
MASA_MAXIMA = 45
Q_MINIMO = 0.25
GIRO_MINIMO = -0.3
GIRO_MAXIMO = 0.4
RETRASO_MAXIMO = 0.010
TOLERANCIA = 0.1
INSTANTE_GLITCH = -5.72
VENTANA_GRAFICO = 0.3
GLITCH_INSTANTE = -45
GLITCH_ANCHO = 0.01
GLITCH_FRECUENCIA = 150
GLITCH_AMPLITUD = 2e-19
COMPUERTA_UMBRAL = 50
COMPUERTA_ANCHO = 0.25
COMPUERTA_TRANSICION = 0.25

In [ ]:
crudos = {}
tiempos = {}
for evento in EVENTOS:
    t0 = gps_evento(evento, DATOS)
    series = {ifo: cargar(ifo, evento, DURACION, DATOS) for ifo in ["H1", "L1"]}
    crudos[evento] = {ifo: serie.value for ifo, serie in series.items()}
    tiempos[evento] = series["H1"].times.value - t0
fs = int(series["H1"].sample_rate.value)
n = DURACION_SEGMENTO * fs
df = fs / n
n_salida = n * FS_SALIDA // fs
indices = np.arange(int(F_MIN / df), int(F_MAX / df) + 1)
frecuencias = np.arange(n // 2 + 1) * df
grados = 2 * BANDAS_CHI2 - 2
banco = np.load(DATOS / "banco.npz")["banco"]
plantilla = generador(indices * df, F_MIN)
print(f"{len(banco)} plantillas, {BANDAS_CHI2} bandas, {grados} grados de libertad")

In [ ]:
def ruido_coloreado(psd, rng):
    espectro = np.sqrt(n * fs * psd / 4) * (rng.standard_normal(n // 2 + 1) + 1j * rng.standard_normal(n // 2 + 1))
    return irfft(espectro)


psd_h1 = psd_en(frecuencias, crudos["GW150914"]["H1"], fs, DURACION_FFT)
muestras = np.arange(BORDE_INICIAL * FS_SALIDA, int((DURACION_SEGMENTO - BORDE_FINAL) * FS_SALIDA), SEPARACION)
rng = np.random.default_rng(0)
chi2_ruido = {}
for masa in MASAS_RUIDO:
    p = banco[np.argmin(np.abs(banco[:, 0] - masa))]
    h = plantilla(*p)
    valores = []
    for _ in range(REALIZACIONES):
        datos_f = transformada(ruido_coloreado(psd_h1, rng), fs, fs // 2)
        valores.append(chi2(datos_f, h, psd_h1, df, n_salida, indices, muestras, BANDAS_CHI2))
    chi2_ruido[masa] = np.concatenate(valores)
    print(f"Mc {p[0]:.1f}: χ²_r medio {chi2_ruido[masa].mean() / grados:.3f} (esperado 1), varianza de χ²_r {np.var(chi2_ruido[masa] / grados):.4f} (esperada {2 / grados:.4f}), {len(chi2_ruido[masa])} muestras")

In [ ]:
producto = np.conj(h) * datos_f[indices] / psd_h1[indices]
rho = serie_snr(datos_f, h[None, :], psd_h1, df, n_salida, indices)[0]
for m in muestras[:5]:
    z = 4 * df * np.sum(producto * np.exp(2j * np.pi * (indices * m % n_salida) / n_salida))
    print(f"muestra {m}: |z|/σ sumando las frecuencias {abs(z) / snr_optimo(h, psd_h1, df, indices):.12f}, serie_snr {rho[m]:.12f}")

In [ ]:
x = np.linspace(0, 3 * grados, 300)
densidad = np.exp((grados / 2 - 1) * np.log(x[1:]) - x[1:] / 2 - grados / 2 * np.log(2) - math.lgamma(grados / 2))
for masa in MASAS_RUIDO:
    plt.hist(chi2_ruido[masa], bins=60, density=True, histtype="step", label=f"Mc {masa}")
plt.plot(x[1:], densidad, "k--", label=f"χ² con {grados} grados")
plt.xlabel("χ²")
plt.ylabel("Densidad")
plt.legend();

In [ ]:
n_total = DURACION * fs
df_total = fs / n_total
indices_total = np.arange(int(F_MIN_INYECCION / df_total), int(F_MAX / df_total) + 1)
en_banda = indices_total * df_total >= F_MIN
forma_inyectada = generador(indices_total * df_total, F_MIN_INYECCION, "IMRPhenomXAS")


def inyectar(x, tiempos, parametros, snr, instante, fase):
    psd = psd_en(np.arange(n_total // 2 + 1) * df_total, x, fs, DURACION_FFT)
    h = forma_inyectada(*parametros)
    escala = snr / snr_optimo(h[en_banda], psd, df_total, indices_total[en_banda])
    espectro = np.zeros(n_total // 2 + 1, dtype=complex)
    espectro[indices_total] = escala * h * np.exp(1j * fase - 2j * np.pi * indices_total * df_total * (instante - tiempos[0]))
    return x + irfft(espectro * fs)

In [ ]:
x = crudos["GW150914"]["H1"]
t = tiempos["GW150914"]
inicio = np.searchsorted(t, INICIO_INYECCIONES)
t_salida = t[inicio] + np.arange(n_salida) / FS_SALIDA
psd_inyecciones = psd_en(frecuencias, x, fs, DURACION_FFT)
rng = np.random.default_rng(1)
for snr in SNR_INYECTADOS:
    reducidos = []
    for _ in range(INYECCIONES_POR_SNR):
        q = rng.uniform(Q_MINIMO, 1)
        parametros = (np.exp(rng.uniform(np.log(MASA_MINIMA), np.log(MASA_MAXIMA))), q / (1 + q) ** 2, rng.uniform(GIRO_MINIMO, GIRO_MAXIMO))
        instante = rng.uniform(INICIO_INYECCIONES + BORDE_INICIAL + MARGEN_INYECCION, INICIO_INYECCIONES + DURACION_SEGMENTO - BORDE_FINAL)
        datos_f = transformada(inyectar(x, t, parametros, snr, instante, rng.uniform(0, 2 * np.pi))[inicio : inicio + n], fs, fs // 2)
        h = plantilla(*parametros)
        rho = serie_snr(datos_f, h[None, :], psd_inyecciones, df, n_salida, indices, np.fft.ifft)[0]
        m = np.argmax(rho * (np.abs(t_salida - instante) < TOLERANCIA))
        reducidos.append(chi2(datos_f, h, psd_inyecciones, df, n_salida, indices, [m], BANDAS_CHI2)[0] / grados)
    print(f"SNR inyectado {snr}: χ²_r en el pico, mediana {np.median(reducidos):.2f} (de {min(reducidos):.2f} a {max(reducidos):.2f})")

In [ ]:
def preparar(evento, inicio_segmento):
    inicio = np.searchsorted(tiempos[evento], inicio_segmento)
    t_salida = tiempos[evento][inicio] + np.arange(n_salida) / FS_SALIDA
    datos_f = {ifo: transformada(x[inicio : inicio + n], fs, fs // 2) for ifo, x in crudos[evento].items()}
    psd = {ifo: psd_en(frecuencias, x, fs, DURACION_FFT) for ifo, x in crudos[evento].items()}
    return t_salida, datos_f, psd


def mejor_coincidencia(datos_f, psd, cerca):
    mejor = (0, 0, 0, 0)
    for a in range(0, len(banco), TANDA):
        plantillas = np.array([plantilla(*p) for p in banco[a : a + TANDA]])
        serie_h1 = serie_snr(datos_f["H1"], plantillas, psd["H1"], df, n_salida, indices, np.fft.ifft)
        serie_l1 = serie_snr(datos_f["L1"], plantillas, psd["L1"], df, n_salida, indices, np.fft.ifft)
        red, desplazamiento = snr_red(serie_h1, serie_l1, int(np.ceil(RETRASO_MAXIMO * FS_SALIDA)))
        red[:, ~cerca] = 0
        q, m = np.unravel_index(red.argmax(), red.shape)
        if red[q, m] > mejor[0]:
            mejor = (red[q, m], a + q, m, desplazamiento[q, m])
    return mejor

In [ ]:
for evento in EVENTOS:
    t_salida, datos_f, psd = preparar(evento, INICIO_EVENTO)
    red, cual, m, k = mejor_coincidencia(datos_f, psd, np.abs(t_salida) < TOLERANCIA)
    h = plantilla(*banco[cual])
    texto = []
    reponderados = []
    for ifo, muestra in [("H1", m), ("L1", m + k)]:
        rho = serie_snr(datos_f[ifo], h[None, :], psd[ifo], df, n_salida, indices, np.fft.ifft)[0, muestra]
        valor_chi2 = chi2(datos_f[ifo], h, psd[ifo], df, n_salida, indices, [muestra], BANDAS_CHI2)[0]
        reponderados.append(snr_reponderado(rho, valor_chi2, BANDAS_CHI2))
        texto.append(f"{ifo} {rho:.1f} (χ²_r {valor_chi2 / grados:.2f}, reponderado {reponderados[-1]:.1f})")
    print(f"{evento}, Mc {banco[cual, 0]:.1f}: " + ", ".join(texto) + f"; red {red:.1f} → reponderada {np.hypot(*reponderados):.1f}")

In [ ]:
t_salida, datos_f, psd = preparar("GW170814", INICIO_EVENTO)
pesada = plantilla(*banco[banco[:, 0].argmax()])
cerca_glitch = np.flatnonzero(np.abs(t_salida - INSTANTE_GLITCH) < VENTANA_GRAFICO)
cerca_evento = np.flatnonzero(np.abs(t_salida) < VENTANA_GRAFICO)
red, cual, m, k = mejor_coincidencia(datos_f, psd, np.abs(t_salida) < TOLERANCIA)
del_evento = plantilla(*banco[cual])
curvas = {}
for nombre, h, cerca in [("glitch en L1", pesada, cerca_glitch), ("GW170814 en L1", del_evento, cerca_evento)]:
    rho = serie_snr(datos_f["L1"], h[None, :], psd["L1"], df, n_salida, indices, np.fft.ifft)[0, cerca]
    valores_chi2 = chi2(datos_f["L1"], h, psd["L1"], df, n_salida, indices, cerca, BANDAS_CHI2)
    curvas[nombre] = (t_salida[cerca], rho, snr_reponderado(rho, valores_chi2, BANDAS_CHI2))
    i = rho.argmax()
    print(f"{nombre}: SNR {rho[i]:.1f}, χ²_r {valores_chi2[i] / grados:.1f}, reponderado {curvas[nombre][2][i]:.1f}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for eje, (nombre, (t, rho, reponderado)) in zip(ejes, curvas.items()):
    eje.plot(t, rho, label="SNR")
    eje.plot(t, reponderado, label="SNR reponderado")
    eje.set_title(nombre)
    eje.set_xlabel("Tiempo desde t0 (s)")
    eje.legend()
ejes[0].set_ylabel("L1");

In [ ]:
x = crudos["GW150914"]["H1"]
t = tiempos["GW150914"]
inicio = np.searchsorted(t, INICIO_INYECCIONES)
tramo = x[inicio : inicio + n].copy()
t_tramo = t[inicio : inicio + n]
tramo += GLITCH_AMPLITUD * np.exp(-((t_tramo - GLITCH_INSTANTE) ** 2) / (2 * GLITCH_ANCHO**2)) * np.sin(2 * np.pi * GLITCH_FRECUENCIA * (t_tramo - GLITCH_INSTANTE))
limpio, centros = compuerta(tramo, fs, COMPUERTA_UMBRAL, COMPUERTA_ANCHO, COMPUERTA_TRANSICION)
print("Compuertas en (s desde t0):", [round(t_tramo[0] + c, 3) for c in centros])
h = plantilla(*banco[np.argmin(np.abs(banco[:, 0] - MASAS_RUIDO[0]))])
t_salida = t_tramo[0] + np.arange(n_salida) / FS_SALIDA
validos = (t_salida > INICIO_INYECCIONES + BORDE_INICIAL) & (t_salida < INICIO_INYECCIONES + DURACION_SEGMENTO - BORDE_FINAL)
sin_compuerta = serie_snr(transformada(tramo, fs, fs // 2), h[None, :], psd_h1, df, n_salida, indices, np.fft.ifft)[0]
con_compuerta = serie_snr(transformada(limpio, fs, fs // 2), h[None, :], psd_h1, df, n_salida, indices, np.fft.ifft)[0]
print(f"SNR máximo en la parte válida: sin compuerta {sin_compuerta[validos].max():.0f}, con compuerta {con_compuerta[validos].max():.1f}")

In [ ]:
plt.semilogy(t_salida[validos], sin_compuerta[validos], label="sin compuerta")
plt.semilogy(t_salida[validos], con_compuerta[validos], label="con compuerta")
plt.xlabel("Tiempo desde t0 (s)")
plt.ylabel("SNR")
plt.legend();